# **Data Cleaning Notebook**

## Objectives

* Evaluate missing data
* Clean data

## Inputs

* outputs/datasets/collection/house_prices_records.csv

## Outputs

* Generate cleaned Train and Test sets, both saved under outputs/datasets/cleaned

## Conclusions

 
Data Cleaning Pipeline

  * Drop: ['EnclosedPorch', 'WoodDeckSF']

  * Impute with median: ['2ndFlrSF', 'BedroomAbvGr', 'LotFrontage', 'MasVnrArea']

  * Impute with constant value -1: ['GarageYrBlt']

  * Impute with mode: ['BsmtExposure', 'BsmtFinType1', 'GarageFinish']
 


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Load Collected data

In [ ]:
import pandas as pd
df_raw_path = "outputs/datasets/collection/house_prices_records.csv"
df = pd.read_csv(df_raw_path)
df.head(3)

---

# Data Exploration

In Data Cleaning we are interested to check the distribution and shape of a variable with missing data.

In [ ]:
vars_with_missing_data = df.columns[df.isna().sum() > 0].to_list()
vars_with_missing_data


In [ ]:
from ydata_profiling import ProfileReport
if vars_with_missing_data:
    profile = ProfileReport(df=df[vars_with_missing_data], minimal=True)
    profile.to_notebook_iframe()
else:
    print("There are no variables with missing data")
    

---

# Correlation and PPS Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ppscore as pps

%matplotlib inline


def heatmap_corr(df, threshold, figsize=(20, 12), font_annot=8):
    """
    Plots a masked heatmap for a given correlation matrix based on a threshold.

    Parameters:
        df (pd.DataFrame): Correlation matrix (Pearson or Spearman).
        threshold (float): Minimum absolute correlation value to display.
        figsize (tuple): Size of the plot.
        font_annot (int): Font size for annotations.
    """
    if len(df.columns) > 1:
        # Mask upper triangle and low-correlation values
        mask = np.zeros_like(df, dtype=bool)
        mask[np.triu_indices_from(mask)] = True
        mask[abs(df) < threshold] = True

        # Plot heatmap
        fig, axes = plt.subplots(figsize=figsize)
        sns.heatmap(df, annot=True, xticklabels=True, yticklabels=True,
                    mask=mask, cmap='viridis',
                    annot_kws={"size": font_annot}, ax=axes,
                    linewidth=0.5)
        axes.set_yticklabels(df.columns, rotation=0)
        plt.ylim(len(df.columns), 0)
        plt.show()


def heatmap_pps(df, threshold, figsize=(20, 12), font_annot=8):
    """
    Plots a masked heatmap for a given PPS (Predictive Power Score) matrix.

    Parameters:
        df (pd.DataFrame): PPS matrix.
        threshold (float): Minimum PPS score to display.
        figsize (tuple): Size of the plot.
        font_annot (int): Font size for annotations.
    """
    if len(df.columns) > 1:
        # Mask low-PPS values
        mask = np.zeros_like(df, dtype=bool)
        mask[abs(df) < threshold] = True

        # Plot heatmap
        fig, ax = plt.subplots(figsize=figsize)
        sns.heatmap(df, annot=True, xticklabels=True, yticklabels=True,
                    mask=mask, cmap='rocket_r',
                    annot_kws={"size": font_annot},
                    linewidth=0.05, linecolor='grey')
        plt.ylim(len(df.columns), 0)
        plt.show()


def CalculateCorrAndPPS(df):
    """
    Calculates Pearson, Spearman correlations and PPS matrix for
    the input DataFrame.

    Parameters:
        df (pd.DataFrame): Input dataset.

    Returns:
        tuple: (Pearson correlation, Spearman correlation, PPS matrix)
    """
    # Compute correlation matrices
    df_corr_spearman = df.corr(method="spearman", numeric_only=True)
    df_corr_pearson = df.corr(method="pearson", numeric_only=True)

    # Compute PPS matrix
    pps_matrix_raw = pps.matrix(df)
    pps_matrix = pps_matrix_raw.filter(['x', 'y', 'ppscore']) \
                               .pivot(columns='x', index='y', values='ppscore')

    # Print statistics for choosing PPS threshold
    pps_score_stats = pps_matrix_raw.query("ppscore < 1") \
                                    .filter(['ppscore']) \
                                    .describe().T
    print("PPS threshold - check PPS score IQR "
          "to decide threshold for heatmap\n")
    print(pps_score_stats.round(3))

    return df_corr_pearson, df_corr_spearman, pps_matrix


def DisplayCorrAndPPS(df_corr_pearson, df_corr_spearman,
                      pps_matrix, CorrThreshold, PPS_Threshold,
                      figsize=(20, 12), font_annot=8):
    """
    Displays heatmaps for Pearson, Spearman correlations and PPS matrix.

    Parameters:
        df_corr_pearson (pd.DataFrame): Pearson correlation matrix.
        df_corr_spearman (pd.DataFrame): Spearman correlation matrix.
        pps_matrix (pd.DataFrame): PPS matrix.
        CorrThreshold (float): Correlation threshold for masking low values.
        PPS_Threshold (float): PPS threshold for masking low values.
        figsize (tuple): Size of the plots.
        font_annot (int): Font size for heatmap annotations.
    """
    print("\n")
    print("* Analyse how the target variable for your ML models are correlated"
          " with other variables (features and target)")
    print("* Analyse multi-colinearity, that is, how the features are "
          "correlated among themselves\n")

    print("*** Heatmap: Spearman Correlation ***")
    print("It evaluates monotonic relationship \n")
    heatmap_corr(df=df_corr_spearman,
                 threshold=CorrThreshold,
                 figsize=figsize,
                 font_annot=font_annot)

    print("*** Heatmap: Pearson Correlation ***")
    print("It evaluates the linear relationship between"
          " two continuous variables \n")
    heatmap_corr(df=df_corr_pearson,
                 threshold=CorrThreshold,
                 figsize=figsize,
                 font_annot=font_annot)

    print("*** Heatmap: Power Predictive Score (PPS) ***")
    print("PPS detects linear or non-linear relationships"
          " between two columns.\n"
          "The score ranges from 0 (no predictive power) to"
          " 1 (perfect predictive power) \n")
    heatmap_pps(df=pps_matrix,
                threshold=PPS_Threshold,
                figsize=figsize,
                font_annot=font_annot)


Calculate Correlations and Power Predictive Score

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
df_corr_pearson, df_corr_spearman, pps_matrix = CalculateCorrAndPPS(df)


Display at Heatmaps

In [ ]:
%matplotlib inline
DisplayCorrAndPPS(df_corr_pearson=df_corr_pearson,
                  df_corr_spearman=df_corr_spearman,
                  pps_matrix=pps_matrix,
                  CorrThreshold=0.4, PPS_Threshold=0.2,
                  figsize=(12, 10), font_annot=10)


---

# Data Cleaning

## Assessing Missing Data Levels

* Custom function to display missing data levels in a DataFrame, it shows the absolute levels, relative levels and data type.

In [ ]:
def EvaluateMissingData(df):
    """
    Evaluates missing data in the provided DataFrame.

    Returns a summary DataFrame showing:
    - The number of rows with missing values per column.
    - The percentage of missing values relative to the total dataset.
    - The data type of each column with missing data.

    Parameters:
        df (pd.DataFrame): The input dataset to check for missing values.

    Returns:
        pd.DataFrame: Summary of columns with missing data,
        sorted by percentage descending.
    """
    # Calculate total number of missing values per column
    missing_data_absolute = df.isnull().sum()

    # Calculate percentage of missing values per column
    missing_data_percentage = round(missing_data_absolute / len(df) * 100, 2)

    # Create a DataFrame with missing info and data types,
    # filter out columns with no missing data
    df_missing_data = (
        pd.DataFrame({
            "RowsWithMissingData": missing_data_absolute,
            "PercentageOfDataset": missing_data_percentage,
            "DataType": df.dtypes
        })
        .sort_values(by="PercentageOfDataset", ascending=False)
        .query("PercentageOfDataset > 0")
    )

    return df_missing_data


Check missing data levels for the collected dataset.

In [ ]:
EvaluateMissingData(df)

## Data Cleaning Spreadsheet Summary

* Drop: ['EnclosedPorch', 'WoodDeckSF']

* Impute with median: ['2ndFlrSF', 'BedroomAbvGr', 'LotFrontage', 'MasVnrArea']

* Impute with constant value -1: ['GarageYrBlt']

* Impute with mode: ['BsmtExposure', 'BsmtFinType1', 'GarageFinish']

**The list above is the guide, and map to know at which stage we are in the data-cleaning process**



### Split Train and Test Set

In [ ]:
from sklearn.model_selection import train_test_split
TrainSet, TestSet, _, __ = train_test_split(
                                        df,
                                        df['SalePrice'],
                                        test_size=0.2,
                                        random_state=0)

print(f"TrainSet shape: {TrainSet.shape} \nTestSet shape: {TestSet.shape}")


In [ ]:
df_missing_data = EvaluateMissingData(TrainSet)
print(f"* There are {df_missing_data.shape[0]} variables with missing data \n")
df_missing_data


## Dealing with Missing Data

In [ ]:
from feature_engine.imputation import (
    MeanMedianImputer,
    CategoricalImputer,
    ArbitraryNumberImputer
)
from feature_engine.selection import DropFeatures
from sklearn.pipeline import Pipeline

# 1. Drop features
drop_vars = ['EnclosedPorch', 'WoodDeckSF']
dropper = DropFeatures(features_to_drop=drop_vars)

# 2. Median imputation
median_vars = ['2ndFlrSF', 'BedroomAbvGr', 'LotFrontage', 'MasVnrArea']
median_imputer = MeanMedianImputer(imputation_method='median',
                                   variables=median_vars)

# 3. Constant value (-1) for numeric "no garage" case
constant_vars = ['GarageYrBlt']
constant_imputer = ArbitraryNumberImputer(arbitrary_number=-1,
                                          variables=constant_vars)

# 4. Mode (most frequent) for categoricals
mode_vars = ['BsmtExposure', 'BsmtFinType1', 'GarageFinish']
mode_imputer = CategoricalImputer(imputation_method='frequent',
                                  variables=mode_vars)


In [ ]:
preprocessing_pipeline = Pipeline([
    ("drop_features", dropper),
    ("median_imputer", median_imputer),
    ("constant_imputer", constant_imputer),
    ("mode_imputer", mode_imputer),
])
preprocessing_pipeline


In [ ]:
preprocessing_pipeline.fit(TrainSet)
df_cleaned = preprocessing_pipeline.transform(TrainSet)


**DataCleaningEffect()**

* Function objective: assess the effect of cleaning the data, when:

    * imput mean, median or arbitrary number is a numerical variable
    * replace with 'Missing' or most frequent a categorical variable
* Parameters: df_original: data not cleaned, df_cleaned: cleaned data,variables_applied_with_method: variables where you applied a given method

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


def DataCleaningEffect(df_original, df_cleaned, variables_applied_with_method):
    """
    Visualizes the effect of data cleaning methods on the
    distribution of specified variables.

    For each variable in the provided list, the function compares
    its distribution in the original
    and cleaned DataFrames using:
    - A count plot for categorical variables
    - A histogram for numerical variables

    Parameters:
        df_original (pd.DataFrame): The original, unmodified dataset.
        df_cleaned (pd.DataFrame): The dataset after cleaning/preprocessing.
        variables_applied_with_method (list):
        List of variable names to visualize.

    Returns:
        None. The function displays the plots.
    """
    flag_count = 1  # Counter for plot titles

    print("\n========================================="
          "============================================")
    print(f"* Distribution Effect Analysis After Data Cleaning"
          " Method in the following variables:")
    print(f"{variables_applied_with_method} \n")

    for var in variables_applied_with_method:
        # Skip variables that were dropped in the cleaned dataset
        if var not in df_cleaned.columns:
            print(f"Skipping {var}: Not found in cleaned dataset.")
            continue

        # Determine if the variable is categorical
        is_categorical = (
            df_original[var].dtype == 'O'
            or df_original[var].nunique() < 10
        )

        if is_categorical:
            # Prepare data for countplot comparison
            df1 = pd.DataFrame({
                "Type": "Original",
                "Value": df_original[var].reset_index(drop=True)
            })
            df2 = pd.DataFrame({
                "Type": "Cleaned",
                "Value": df_cleaned[var].reset_index(drop=True)
            })
            dfAux = pd.concat([df1, df2], axis=0).reset_index(drop=True)

            # Plot countplot for categorical variable
            fig, ax = plt.subplots(figsize=(15, 5))
            sns.countplot(
                data=dfAux, x="Value", hue="Type",
                palette=['#432371', "#FAAE7B"], ax=ax
            )
            ax.set_title(f"Distribution Plot {flag_count}: {var}")
            plt.xticks(rotation=45)
            plt.legend()

        else:
            # Plot histogram for numerical variable
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.histplot(
                df_original[var], kde=True, element="step",
                color="#432371", label="Original", ax=ax
            )
            sns.histplot(
                df_cleaned[var], kde=True, element="step",
                color="#FAAE7B", label="Cleaned", ax=ax
            )
            ax.set_title(f"Distribution Plot {flag_count}: {var}")
            plt.legend()

        # Adjust layout and show plot
        plt.tight_layout()
        plt.show()
        flag_count += 1


In [ ]:
variables_applied_with_method = [
    '2ndFlrSF', 'BedroomAbvGr', 'LotFrontage', 'MasVnrArea',  # median
    'GarageYrBlt',                                      # constant -1
    'BsmtExposure', 'BsmtFinType1', 'GarageFinish'      # mode
]

DataCleaningEffect(TrainSet, df_cleaned, variables_applied_with_method)


**Final Review of Cleaning Decisions**

After analyzing pre- and post-cleaning distributions, we concluded that most imputation methods preserved the original feature structure and distribution well. The following observations were made:

* Most features (e.g., MasVnrArea, BsmtFinType1, GarageFinish, etc.) were cleaned effectively, with imputations aligning well with domain logic and minimal distortion to their distributions.

* GarageYrBlt showed some distortion due to missing values being replaced with 0, which is not a valid year. This led to an artificial spike at zero. We'll retain the current imputation for now but may consider outlier filtering or replacing zeros with NaN and imputing differently in future iterations.

* LotFrontage initially used median imputation due to its simplicity and robustness against outliers. However, correlation and distributional analyses suggested this variable might benefit from predictive imputation techniques like KNN. We’ll keep the median approach for now and revisit it if model performance indicates a need for refinement.

This balanced approach helps preserve data integrity while keeping our pipeline simple and transparent. Any potentially suboptimal imputations are documented and will be reconsidered during the modeling evaluation phase.



If you are satisfied, apply the transformation to your data.

In [ ]:
# Step 1: Fit the pipeline on the training data and transform it
TrainSet_processed = preprocessing_pipeline.fit_transform(TrainSet)

# Step 2: Transform the test data using the already-fitted pipeline
TestSet_processed = preprocessing_pipeline.transform(TestSet)


In [ ]:
import pandas as pd

# Get the correct column names by fitting only the dropper step
remaining_features = dropper.fit(TrainSet).transform(TrainSet).columns

# Convert arrays to DataFrames using actual transformed columns
TrainSet_cleaned = pd.DataFrame(TrainSet_processed,
                                columns=remaining_features,
                                index=TrainSet.index)
TestSet_cleaned = pd.DataFrame(TestSet_processed,
                               columns=remaining_features,
                               index=TestSet.index)


Evaluate if you have more variables to deal with. If yes, iterate. If not, you are done.

In [ ]:
EvaluateMissingData(TrainSet_cleaned)

---

# Push files to Repo

* If you do not need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
try:
    # create outputs/datasets/collection folder
    os.makedirs(name='outputs/datasets/cleaned')
except Exception as e:
    print(e)



## Train Set

In [ ]:
TrainSet_cleaned.to_csv(
    "outputs/datasets/cleaned/TrainSetCleaned.csv", index=False)


## Test Set

In [ ]:
TestSet_cleaned.to_csv(
    "outputs/datasets/cleaned/TestSetCleaned.csv", index=False)
